In [9]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import evaluate

import torch
print("torch:", torch.__version__, "cuda_available:", torch.cuda.is_available())

torch: 2.8.0+cu128 cuda_available: True


In [10]:
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

categories = {
    "greeting": [
        "こんにちは", "おはようございます", "こんばんは", "やあ元気ですか？", "もしもし", "お疲れ様です", "どうも"
    ],
    "goodbye": [
        "さようなら", "またね", "じゃあね", "バイバイ", "おやすみなさい", "ではまた"
    ],
    "thanks": [
        "ありがとう", "どうもありがとうございます", "感謝します", "本当に助かりました", "ありがとうございます！"
    ],
    "apology": [
        "すみません", "ごめんなさい", "申し訳ありません", "ご迷惑をおかけしました"
    ],
    "price": [
        "いくらですか？", "これはいくらですか？", "お値段は？", "高いですね", "安いですね"
    ],
    "food": [
        "美味しいです", "まずいです", "コーヒーをください", "水をください", "ご飯はありますか？"
    ],
    "location": [
        "トイレはどこですか？", "駅はどこですか？", "この近くにレストランはありますか？", "図書館はどこですか？"
    ]
}

# Generate more realistic dataset
def generate_dataset(n_samples=50000):
    data = []
    for _ in range(n_samples):
        label = random.choice(list(categories.keys()))
        text = random.choice(categories[label])

        # Randomly add punctuation/particle
        if random.random() > 0.5:
            text += random.choice(["！", "？", "ね", "よ"])

        # Randomly mix Kanji/Katakana/Hiragana by adding extra phrases
        if random.random() > 0.7:
            extra = random.choice(sum(categories.values(), []))
            text += " " + extra

        data.append([text, label])
    return pd.DataFrame(data, columns=["text", "label"])

df = generate_dataset(50000)

le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'], df['label_encoded'], test_size=0.2, random_state=42, stratify=df['label_encoded']
)

print("Samples:", len(df))
print("Train size:", len(train_texts), "Val size:", len(val_texts))


Samples: 50000
Train size: 40000 Val size: 10000


In [11]:
from janome.tokenizer import Tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer

tokenizer = Tokenizer()

# Tokenization function
def tokenize_japanese(text):
    return " ".join([token.surface for token in tokenizer.tokenize(text)])

# Apply tokenization
df['tokenized'] = df['text'].apply(tokenize_japanese)

# TF-IDF vectorization
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['tokenized'])
y = df['label']

print("Tokenization and TF-IDF done!")
print("Feature matrix shape:", X.shape)

Tokenization and TF-IDF done!
Feature matrix shape: (50000, 45)


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "LogisticRegression": LogisticRegression(max_iter=200),
    "NaiveBayes": MultinomialNB(),
    "SVM": SVC()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"\n{name} Results:")
    print(classification_report(y_test, y_pred))



LogisticRegression Results:
              precision    recall  f1-score   support

     apology       0.88      0.96      0.91      1389
        food       0.89      0.86      0.87      1409
     goodbye       0.92      0.85      0.88      1456
    greeting       0.95      0.81      0.88      1414
    location       0.85      0.96      0.90      1455
       price       0.88      0.87      0.88      1449
      thanks       0.86      0.90      0.88      1428

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


NaiveBayes Results:
              precision    recall  f1-score   support

     apology       0.90      0.90      0.90      1389
        food       0.87      0.86      0.87      1409
     goodbye       0.89      0.86      0.88      1456
    greeting       0.89      0.85      0.87      1414
    location       0.89      0.89      0.89      1455
       price       0.86     

In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("cl-tohoku/bert-base-japanese")

# Tokenize dataset
train_encodings = tokenizer(list(train_texts), padding=True, truncation=True, return_tensors="pt")
val_encodings = tokenizer(list(val_texts), padding=True, truncation=True, return_tensors="pt")

train_labels = torch.tensor(train_labels.values)
val_labels = torch.tensor(val_labels.values)

# Dataset class
class JapaneseDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = JapaneseDataset(train_encodings, train_labels)
val_dataset = JapaneseDataset(val_encodings, val_labels)

# Model
model = AutoModelForSequenceClassification.from_pretrained(
    "cl-tohoku/bert-base-japanese",
    num_labels=df['label_encoded'].nunique()
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cl-tohoku/bert-base-japanese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Define metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

# Training args with eval
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",   # ✅ FIXED
    logging_dir="./logs",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.evaluate()

trainer.save_model("./results")         
tokenizer.save_pretrained("./results")

import joblib
joblib.dump(le, "./results/label_encoder.pkl")


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.001200,0.002587,0.999700,0.999700
2,0.000000,0.001558,0.999800,0.999800
3,0.000100,0.001177,0.999800,0.999800


['./results/label_encoder.pkl']

In [15]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BertForSequenceClassification, BertConfig
import torch
import numpy as np
import os

# 1️⃣ Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("cl-tohoku/bert-base-japanese")

# 2️⃣ Load model
if os.path.exists("./results/config.json"):
    # ✅ Case 1: Saved with trainer.save_model("./results")
    print("Loading model from Hugging Face format...")
    model = AutoModelForSequenceClassification.from_pretrained("./results")
    tokenizer = AutoTokenizer.from_pretrained("./results")
elif os.path.exists("./results/model.pth"):
    # ✅ Case 2: Saved only state_dict
    print("Loading model from state_dict...")
    config = BertConfig.from_pretrained("cl-tohoku/bert-base-japanese", num_labels=4)  # ⚠️ change num_labels
    model = BertForSequenceClassification(config)
    model.load_state_dict(torch.load("./results/model.pth", map_location="cpu"))
    model.eval()
else:
    raise ValueError("❌ No valid model found in ./results. Did you save it correctly?")

# 3️⃣ If you had a LabelEncoder during training, reload it
import joblib
if os.path.exists("./results/label_encoder.pkl"):
    le = joblib.load("./results/label_encoder.pkl")
else:
    le = None

# 4️⃣ Prediction function
def predict_label(text):
    # Tokenize
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    
    # Move to GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    
    # Get predicted label
    pred_id = torch.argmax(logits, dim=1).item()
    if le:
        return le.inverse_transform([pred_id])[0]
    else:
        return pred_id  # return numeric label if no LabelEncoder

# 5️⃣ Test examples
sentences = [
    "こんにちは", 
    "すみません、道を教えてください", 
    "これはいくらですか？", 
    "美味しいです！"
]

for s in sentences:
    print(f"Text: {s} → Predicted Label: {predict_label(s)}")


Loading model from Hugging Face format...
Text: こんにちは → Predicted Label: greeting
Text: すみません、道を教えてください → Predicted Label: apology
Text: これはいくらですか？ → Predicted Label: price
Text: 美味しいです！ → Predicted Label: food
